# Avance 5. Modelo final: comparación de ensambles y evaluación en datos no vistos

**Inferencia de propiedades físicas de galaxias a partir de imágenes multibanda** · Instituto Tecnológico y de Estudios Superiores de Monterrey · Maestría en Inteligencia Artificial Aplicada · **Equipo 24** · Colaboración con el Instituto de Astronomía de la UNAM

| Integrante | Matrícula |
|---|---|
| Luz Copelia Minutti Pérez | A01796921 |
| Erick Alberto Bustos Cruz | A01378966 |
| José Florencio Maguey Peralta | A01796727 |

**Asesores:** Dr. Iván Reyes Amezcua (ITESM) · Dr. Héctor Hernández Toledo y Dr. José Antonio Vázquez Mata (IA-UNAM)

**Fase de CRISP-ML(Q):** 3, *Machine Learning Model Engineering* (Studer et al. 2021). **Avance 5:** modelo final.

## 1. Planteamiento

En el Avance 4 comparamos seis modelos individuales y vimos que ninguno domina en todas las propiedades: la ResNet50 con redshift estima mejor la masa, y DINOv3 con LoRA estima mejor la SFR y el D4000. En este avance construimos ensambles para aprovechar esa complementariedad, de dos tipos. Los **homogéneos** combinan tres redes de la misma arquitectura entrenadas con semillas distintas; los **heterogéneos** combinan arquitecturas distintas. Este cuaderno reúne todos los modelos, los compara con los mismos criterios, elige el modelo final y lo evalúa en el conjunto de prueba.

El trabajo está repartido en cuatro cuadernos de esta carpeta:

| Cuaderno | Contenido |
|---|---|
| `Ensambles_Homogeneos_ResNet50_D_E_vf.ipynb` | Ensambles homogéneos de ResNet50 (variantes D y E, tres semillas cada una) y búsqueda de sus pesos, evaluados en validación |
| `Modelos_Base_DINOv3_ConvNeXt_EfficientNet.ipynb` | Entrenamiento en GPU de los miembros nuevos y exportación de las predicciones de nuestras redes |
| `Ensambles_Heterogeneos_Mezcla_MetaAprendiz.ipynb` | Ensambles heterogéneos: mezcla convexa y meta-aprendiz, con su optimización y su validación anidada |
| `Avance5.24.ipynb` | Comparación final de los modelos individuales y de los ensambles, selección del modelo final y evaluación en prueba |

Los cuatro ensambles que pide el avance son de dos tipos. Los homogéneos son los dos de ResNet50 de Luz Minutti (variantes D y E) y los de nuestras tres familias con redshift (DINOv3, ConvNeXt y EfficientNet-B3, tres semillas cada una). Los heterogéneos son la mezcla convexa y el meta-aprendiz. Los ensambles de Luz se reportan con las métricas de validación de su cuaderno: no contamos con sus predicciones galaxia por galaxia ni con predicciones de prueba, así que entran a la comparación como referencia, pero no a la regla de selección ni a la evaluación en prueba.

Ningún cuaderno anterior calculó métricas con el conjunto de prueba. Aquí elegimos el modelo final con las galaxias de validación, registramos la elección con una huella de sus predicciones y solo después abrimos la prueba, una sola vez.

**El interruptor de la pista.** La pista `con_redshift` usa modelos que reciben el redshift espectroscópico como entrada, igual que la variante E de ResNet50; la pista `solo_imagen` usa modelos que solo reciben la imagen, que es la situación de la mayoría de las galaxias a las que queremos extender las estimaciones. Con `PISTA = "con_redshift"` (valor por defecto) se compara la primera y al cambiar la variable se compara la segunda. El cuaderno de ensambles heterogéneos debe haberse ejecutado antes con la misma pista.

## 2. Parámetros

| Parámetro | Valor | Significado |
|---|---|---|
| `PISTA` | `con_redshift` | Interruptor de la pista: `con_redshift` o `solo_imagen` |
| `SEMILLA_REPRESENTANTE` | 42 | Semilla de la red que representa a cada familia como modelo individual, fijada antes de ver resultados |
| `N_REMUESTREOS` | 2000 | Remuestreos *bootstrap* para los intervalos de confianza |

| Pista | Familias que se combinan en los heterogéneos |
|---|---|
| `con_redshift` | `dinov3_z`, `convnext_z`, `efficientnet_z` |
| `solo_imagen` | `dinov3_imagen`, `convnext_imagen` y, si se entrenó, `efficientnet_imagen` |

Las redes de ResNet50 del modelo 1 no entran a los heterogéneos: el cuaderno de ensambles homogéneos reporta sus métricas de validación, pero no exporta sus predicciones galaxia por galaxia ni predice el conjunto de prueba, y sin esas predicciones no se pueden combinar con otras redes.

In [ ]:
import os
variable_entorno = os.environ.get          # las variables MA_* solo se usan en la prueba local reducida

# ------------------------------------------------------------------------------------------------
# INTERRUPTOR DE LA PISTA: "con_redshift" (redshift espectroscópico como entrada) o "solo_imagen".
PISTA = "con_redshift"
# ------------------------------------------------------------------------------------------------
PISTA = variable_entorno("MA_PISTA", PISTA)
PISTAS = {"con_redshift": {"heterogeneo": ["dinov3_z", "convnext_z", "efficientnet_z"], "otros": [],
                           "redshift_en_meta": True, "descripcion": "con el redshift espectroscópico como entrada"},
          "solo_imagen": {"heterogeneo": ["dinov3_imagen", "convnext_imagen", "efficientnet_imagen"], "otros": [],
                          "redshift_en_meta": False, "descripcion": "solo con la imagen"}}
if PISTA not in PISTAS:
    raise ValueError(f"PISTA debe ser una de {list(PISTAS)}")
OBJETIVOS = ("log_mstar", "log_sfr", "d4000")
NOMBRES = {"log_mstar": "log M* [dex]", "log_sfr": "log SFR [dex]", "d4000": "D4000"}
CORTO = {"log_mstar": "M*", "log_sfr": "SFR", "d4000": "D4000"}
NOMBRE_H1, NOMBRE_H2 = "Heterogéneo 1: mezcla convexa", "Heterogéneo 2: meta-aprendiz"
ARCHIVO_H1, ARCHIVO_H2 = "h1_mezcla_convexa", "h2_meta_aprendiz"

SEMILLA_REPRESENTANTE = 42
N_REMUESTREOS = int(variable_entorno("MA_REMUESTREOS", "2000"))
print(f"Pista: {PISTA} ({PISTAS[PISTA]['descripcion']})")

## 3. Entorno

Este cuaderno no necesita GPU ni acceso a las imágenes: trabaja con las predicciones que cada red exportó en un formato común (un archivo pequeño por familia y semilla, con las galaxias de validación y de prueba), que están en el repositorio. Descargamos el código del equipo para ligar los resultados a una versión concreta.

In [ ]:
import sys, json, time, subprocess, platform, warnings, hashlib
from pathlib import Path
from datetime import datetime, timezone

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/erteck/InferenciaPropiedadesFisicas.git"
CARPETA_REPOSITORIO = Path(variable_entorno("MA_CARPETA_REPOSITORIO", "/content/InferenciaPropiedadesFisicas"))
if EN_COLAB:
    if not CARPETA_REPOSITORIO.exists():
        subprocess.run(["git", "clone", "-q", URL_REPOSITORIO, str(CARPETA_REPOSITORIO)], check=True)
    subprocess.run(["git", "-C", str(CARPETA_REPOSITORIO), "pull", "-q"], check=True)
elif not (CARPETA_REPOSITORIO / "src").exists():
    CARPETA_REPOSITORIO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "modelos_alternativos").exists())
sys.path.insert(0, str(CARPETA_REPOSITORIO))
COMMIT = subprocess.run(["git", "-C", str(CARPETA_REPOSITORIO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display
from src.modelos_alternativos import ensambles, evaluacion, predicciones

CARPETA_PREDICCIONES = Path(variable_entorno("MA_CARPETA_PREDICCIONES", str(CARPETA_REPOSITORIO / "Avance 5. Modelo final" / "predicciones")))
CARPETA_ENTREGA = CARPETA_PREDICCIONES.parent
CARPETA_HETEROGENEOS = CARPETA_ENTREGA / "ensambles_heterogeneos"
pd.set_option("display.precision", 4, "display.max_columns", 40, "display.width", 220)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
print("Python", platform.python_version(), "· commit", COMMIT[:12], "·", CARPETA_PREDICCIONES)

## 4. Predicciones de los modelos

Leemos las predicciones de todas las redes y las alineamos por el identificador de cada galaxia (`mangaid`), nunca por posición, con la misma regla que el cuaderno de ensambles heterogéneos: si alguna red no trae predicción para una galaxia, esa galaxia se descarta para todas. También se comprueba que las etiquetas y el redshift coincidan entre archivos. La métrica principal, el promedio entre propiedades del MSE normalizado por la desviación de entrenamiento (cuanto menor, mejor), usa una escala que viene en un archivo aparte porque este cuaderno no tiene acceso al catálogo.

In [ ]:
TABLA = predicciones.leer_carpeta(CARPETA_PREDICCIONES)
ESCALA = predicciones.leer_escala(CARPETA_PREDICCIONES)
CONFIGURACION = PISTAS[PISTA]
CANDIDATAS = CONFIGURACION["heterogeneo"] + CONFIGURACION["otros"]
DISPONIBLES = [f for f in CANDIDATAS if f in set(TABLA[TABLA.particion == "Validation"].familia) and f in set(TABLA[TABLA.particion == "Test"].familia)]
FALTAN = [f for f in CANDIDATAS if f not in DISPONIBLES]
if FALTAN:
    print("Aviso: no hay predicciones de validación y de prueba para:", FALTAN, "· se trabaja sin ellas")
FAMILIAS_HETEROGENEO = [f for f in CONFIGURACION["heterogeneo"] if f in DISPONIBLES]
if len(FAMILIAS_HETEROGENEO) < 2:
    raise ValueError("Un ensamble heterogéneo necesita al menos dos arquitecturas con predicciones")

# Las galaxias son las que tienen predicción en todas las familias disponibles, igual en los dos cuadernos sin GPU.
VAL = predicciones.construir_conjunto(TABLA, "Validation", DISPONIBLES)
PRUEBA = predicciones.construir_conjunto(TABLA, "Test", DISPONIBLES)
Y_VAL, Y_PRUEBA, Z_VAL, Z_PRUEBA = VAL.y, PRUEBA.y, VAL.redshift, PRUEBA.redshift
P_VAL = VAL.medias(FAMILIAS_HETEROGENEO)              # (familias, galaxias, propiedades): promedio de las semillas
P_PRUEBA = PRUEBA.medias(FAMILIAS_HETEROGENEO)
HUELLA_MIEMBROS = predicciones.huella_miembros(VAL, FAMILIAS_HETEROGENEO)
resumen_carga = pd.DataFrame({"semillas": {f: len(VAL.semillas[f]) for f in DISPONIBLES},
                              "galaxias de validación": len(VAL.ids), "galaxias de prueba": len(PRUEBA.ids),
                              "tta": TABLA.groupby("familia").tta.first().reindex(DISPONIBLES).map({True: "sí", False: "no"}),
                              "en los heterogéneos": {f: "sí" if f in FAMILIAS_HETEROGENEO else "no" for f in DISPONIBLES}})
display(resumen_carga)
print("Desviación de cada propiedad en entrenamiento:", dict(zip(OBJETIVOS, ESCALA.round(4))))

Después leemos los dos ensambles heterogéneos. Antes de usarlos comprobamos que se ajustaron con la misma pista, las mismas familias y exactamente las mismas predicciones de los miembros que acabamos de leer, comparando la huella que guardó su cuaderno con la que calculamos aquí. Si alguna red se volvió a exportar después de ajustar los ensambles, la huella no coincide y el cuaderno se detiene.

In [ ]:
RUTA_CONFIGURACION = CARPETA_HETEROGENEOS / f"heterogeneos_{PISTA}.json"
if not RUTA_CONFIGURACION.exists():
    raise FileNotFoundError(f"No existe {RUTA_CONFIGURACION}: ejecuta antes Ensambles_Heterogeneos_Mezcla_MetaAprendiz.ipynb con PISTA = '{PISTA}'")
HETEROGENEOS = json.loads(RUTA_CONFIGURACION.read_text())
if HETEROGENEOS["pista"] != PISTA or HETEROGENEOS["familias"] != FAMILIAS_HETEROGENEO:
    raise ValueError(f"Los ensambles heterogéneos se ajustaron con la pista {HETEROGENEOS['pista']} y las familias {HETEROGENEOS['familias']}")
if HETEROGENEOS["huella_miembros"] != HUELLA_MIEMBROS:
    raise ValueError("Las predicciones de los miembros cambiaron desde que se ajustaron los ensambles heterogéneos: vuelve a ejecutar ese cuaderno")
OOF, PRED_PRUEBA_HETEROGENEO = {}, {}
for archivo, nombre in ((ARCHIVO_H1, NOMBRE_H1), (ARCHIVO_H2, NOMBRE_H2)):
    OOF[nombre], PRED_PRUEBA_HETEROGENEO[nombre] = predicciones.leer_ensamble(CARPETA_HETEROGENEOS, PISTA, archivo, VAL.ids, PRUEBA.ids)
PESOS_H1 = np.asarray(HETEROGENEOS[ARCHIVO_H1]["pesos"], float)
APORTE = pd.read_csv(CARPETA_HETEROGENEOS / f"aporte_familias_{PISTA}.csv")
display(pd.DataFrame({nombre: {"configuración": {k: v for k, v in HETEROGENEOS[a]["configuracion"].items()},
                               "criterio de la validación anidada": f"{HETEROGENEOS[a]['criterio_medio']:.4f} ± {HETEROGENEOS[a]['criterio_desviacion']:.4f}",
                               "repeticiones": len(OOF[nombre]), "ajustado el": HETEROGENEOS["fecha_utc"][:19], "commit": HETEROGENEOS["commit"][:12]}
                      for a, nombre in ((ARCHIVO_H1, NOMBRE_H1), (ARCHIVO_H2, NOMBRE_H2))}).T)
print("Huella de los miembros verificada:", HUELLA_MIEMBROS[:16])

Para el costo de cada modelo juntamos los tiempos que registró el cuaderno de entrenamiento de los modelos base, todos medidos en la misma GPU A100: los minutos de entrenamiento de cada red y los milisegundos por galaxia al predecir sin TTA. Ese cuaderno midió también el tiempo de uso de la ResNet50 con la misma arquitectura y el mismo tamaño de imagen (la duración de una predicción no depende de los valores de los pesos), para poder comparar el costo de usar los ensambles de Luz. Sus tiempos de entrenamiento son los que ella reportó en una GPU NVIDIA RTX A4000, más lenta que la A100, así que anotamos siempre el equipo.

In [ ]:
archivos_tiempos = sorted(CARPETA_PREDICCIONES.glob("tiempos_modelos*.csv"))
TIEMPOS = pd.concat([pd.read_csv(a) for a in archivos_tiempos], ignore_index=True) if archivos_tiempos else pd.DataFrame(
    columns=["familia", "semilla", "hardware", "minutos_entrenamiento", "ms_por_galaxia"])
primero_valido = lambda x: x.dropna().iloc[0] if x.notna().any() else np.nan
entrenamiento = TIEMPOS[TIEMPOS.minutos_entrenamiento.notna()]                # filas de entrenamiento (las de solo inferencia no cuentan)
COSTO_FAMILIA = pd.DataFrame({"minutos_por_red": entrenamiento.groupby("familia").minutos_entrenamiento.mean(),
                              "hardware": entrenamiento.groupby("familia").hardware.first(),
                              "ms_por_galaxia": TIEMPOS.groupby("familia").ms_por_galaxia.agg(primero_valido)})
MS_RESNET50 = COSTO_FAMILIA.ms_por_galaxia.get("resnet50_e" if PISTA == "con_redshift" else "resnet50_g", np.nan)

def costo_de(familias, redes_por_familia=None):
    # Costo de un modelo que ejecuta `redes_por_familia` redes de cada familia (por defecto, todas sus semillas).
    n = {f: (redes_por_familia or len(VAL.semillas[f])) for f in familias}
    minutos = sum(n[f] * COSTO_FAMILIA.minutos_por_red.get(f, np.nan) for f in familias)
    ms = sum(n[f] * COSTO_FAMILIA.ms_por_galaxia.get(f, np.nan) for f in familias)
    equipos = sorted({h if isinstance(h, str) else "no reportado" for h in (COSTO_FAMILIA.hardware.get(f, np.nan) for f in familias)})
    return {"redes por galaxia": sum(n.values()), "minutos de entrenamiento": minutos, "ms por galaxia": ms,
            "horas de GPU por millón": ensambles.horas_gpu_por_millon(ms) if np.isfinite(ms) else np.nan, "equipo": " + ".join(equipos)}
display(COSTO_FAMILIA.reindex(DISPONIBLES).round(3).fillna("no reportado"))

## 5. Modelos individuales y ensambles homogéneos

Cada familia se representa de dos formas. Como **modelo individual** usamos la red de la semilla 42, la misma que representó a la variante E de ResNet50 en el Avance 4, y la fijamos antes de ver los resultados para no elegir la semilla que mejor salió. Como **ensamble homogéneo** usamos el promedio uniforme de sus tres semillas. Luz Minutti buscó además pesos aprendidos para sus ensambles de ResNet50. En la variante E la búsqueda eligió el propio promedio uniforme, y en la D eligió pesos por propiedad con λ = 0.01, cuya ventaja dentro de la búsqueda fue de una diezmilésima (0.11899 contra 0.11910). Por eso usamos el promedio uniforme en nuestras familias: no tiene hiperparámetros y no pierde nada que se distinga del ruido.

La tabla ordena nuestras redes y sus ensambles homogéneos por la métrica principal en validación. En estos modelos no se ajusta nada con esas galaxias, así que sus valores son directamente comparables. Nuestras predicciones promedian las 8 simetrías de cada imagen (TTA), que en el Avance 4 redujo el criterio entre 1.4 y 4 %; las cifras de Luz son sin TTA, y lo tenemos en cuenta al compararlas.

In [ ]:
def metricas_largas(nombre, y, prediccion, **extra):
    tabla = evaluacion.tabla_metricas(y, prediccion, OBJETIVOS, ESCALA).set_index("objetivo")
    return {"modelo": nombre, **extra, "criterio": tabla.MSE_normalizado.mean(),
            **{f"RMSE {CORTO[t]}": tabla.loc[t, "RMSE"] for t in OBJETIVOS}, **{f"MAE {CORTO[t]}": tabla.loc[t, "MAE"] for t in OBJETIVOS},
            **{f"R2 {CORTO[t]}": tabla.loc[t, "R2"] for t in OBJETIVOS}}

def indice_representante(familia):
    semillas = VAL.semillas[familia]
    return semillas.index(SEMILLA_REPRESENTANTE) if SEMILLA_REPRESENTANTE in semillas else 0

INDIVIDUAL_VAL = {f: VAL.pred[f][indice_representante(f)] for f in DISPONIBLES}
INDIVIDUAL_PRUEBA = {f: PRUEBA.pred[f][indice_representante(f)] for f in DISPONIBLES}
HOMOGENEO_VAL = {f: VAL.media(f) for f in DISPONIBLES}
HOMOGENEO_PRUEBA = {f: PRUEBA.media(f) for f in DISPONIBLES}

filas = []
for f in DISPONIBLES:
    por_semilla = [ensambles.criterio(Y_VAL, VAL.pred[f][i], ESCALA) for i in range(len(VAL.semillas[f]))]
    filas.append({**metricas_largas(f"{f} · individual (semilla {VAL.semillas[f][indice_representante(f)]})", Y_VAL, INDIVIDUAL_VAL[f], tipo="individual", familia=f),
                  "criterio medio de las semillas": np.mean(por_semilla), "desviación entre semillas": np.std(por_semilla, ddof=1) if len(por_semilla) > 1 else np.nan})
    filas.append(metricas_largas(f"{f} · ensamble homogéneo ({len(VAL.semillas[f])} semillas)", Y_VAL, HOMOGENEO_VAL[f], tipo="homogéneo", familia=f))
BASE = pd.DataFrame(filas).sort_values("criterio").reset_index(drop=True)
display(BASE[["modelo", "criterio", "criterio medio de las semillas", "desviación entre semillas", "RMSE M*", "RMSE SFR", "RMSE D4000", "R2 M*", "R2 SFR", "R2 D4000"]]
        .round(4).fillna("no aplica"))
MEJOR_INDIVIDUAL = BASE[BASE.tipo == "individual"].iloc[0].familia
print("Mejor modelo individual (semilla representante):", MEJOR_INDIVIDUAL)

_Se completa con los resultados de la corrida._

## 6. Comparación en validación

Reunimos el mejor modelo individual, los ensambles homogéneos y los dos heterogéneos, ordenados por la métrica principal, con el RMSE, el MAE y el R² de cada propiedad y con el costo de cada uno. Para los heterogéneos usamos sus predicciones fuera de pliegue de la validación anidada, de modo que ninguna galaxia fue predicha por un combinador ajustado con su propia etiqueta; su criterio es el promedio de las repeticiones y su desviación indica cuánto cambia con la partición. Las métricas por propiedad de los heterogéneos son las de la primera repetición.

El costo tiene tres partes. Los minutos de entrenamiento suman el de todas las redes del modelo, sin contar las búsquedas de hiperparámetros ni el ajuste del combinador, que dura segundos. Las redes por galaxia y los milisegundos por galaxia (sin TTA, en la A100) miden lo que cuesta usarlo, y de ahí extrapolamos las horas de GPU para un millón de galaxias. Un ensamble necesita ejecutar todas sus redes, así que su costo de uso es la suma de los de sus miembros; las predicciones que combinamos usan TTA de 8 simetrías, que multiplicaría ese costo por ocho si se conserva al desplegar.

La tabla incluye, en amarillo en la gráfica, los ensambles homogéneos de Luz con las métricas de validación que reporta su cuaderno. Sirven como referencia, pero no entran a la regla de selección ni a la evaluación en prueba, porque para ambas se necesitan las predicciones de cada galaxia.

In [ ]:
NOMBRE_INDIVIDUAL = f"Individual: {MEJOR_INDIVIDUAL} (mejor red individual)"
nombre_homogeneo = lambda f: f"Homogéneo: {f} ({len(VAL.semillas[f])} semillas)"
FAMILIAS_ACTIVAS_H1 = [f for i, f in enumerate(FAMILIAS_HETEROGENEO) if PESOS_H1[i].max() > 0.01]

PRED_VAL, PRED_PRUEBA, FAMILIA_DE = {NOMBRE_INDIVIDUAL: INDIVIDUAL_VAL[MEJOR_INDIVIDUAL]}, {NOMBRE_INDIVIDUAL: INDIVIDUAL_PRUEBA[MEJOR_INDIVIDUAL]}, {}
PERDIDAS = {NOMBRE_INDIVIDUAL: ensambles.perdida_por_galaxia(Y_VAL, INDIVIDUAL_VAL[MEJOR_INDIVIDUAL], ESCALA)}
FILAS = [{**metricas_largas(NOMBRE_INDIVIDUAL, Y_VAL, INDIVIDUAL_VAL[MEJOR_INDIVIDUAL], tipo="individual"), "desviación entre repeticiones": np.nan,
          **costo_de([MEJOR_INDIVIDUAL], redes_por_familia=1), "detalle": "una red"}]
for f in DISPONIBLES:
    n = nombre_homogeneo(f)
    PRED_VAL[n], PRED_PRUEBA[n] = HOMOGENEO_VAL[f], HOMOGENEO_PRUEBA[f]
    PERDIDAS[n] = ensambles.perdida_por_galaxia(Y_VAL, HOMOGENEO_VAL[f], ESCALA)
    FILAS.append({**metricas_largas(n, Y_VAL, HOMOGENEO_VAL[f], tipo="homogéneo"), "desviación entre repeticiones": np.nan, **costo_de([f]),
                  "detalle": "promedio uniforme de las semillas"})
for archivo, nombre, familias in ((ARCHIVO_H1, NOMBRE_H1, FAMILIAS_ACTIVAS_H1), (ARCHIVO_H2, NOMBRE_H2, FAMILIAS_HETEROGENEO)):
    PRED_VAL[nombre], PRED_PRUEBA[nombre] = OOF[nombre][0], PRED_PRUEBA_HETEROGENEO[nombre]
    PERDIDAS[nombre] = np.mean([ensambles.perdida_por_galaxia(Y_VAL, o, ESCALA) for o in OOF[nombre]], axis=0)   # promedio de las repeticiones
    config = HETEROGENEOS[archivo]["configuracion"]
    detalle = (f"λ={config['lambda']}, pesos {config['alcance'].replace('_', ' ')}" if archivo == ARCHIVO_H1 else f"tipo {config['tipo']}")
    FILAS.append({**metricas_largas(nombre, Y_VAL, OOF[nombre][0], tipo="heterogéneo"), "criterio": float(PERDIDAS[nombre].mean()),
                  "desviación entre repeticiones": HETEROGENEOS[archivo]["criterio_desviacion"], **costo_de(familias),
                  "detalle": f"{detalle}; ajuste del combinador {HETEROGENEOS[archivo]['segundos_ajuste_final']:.1f} s"})
COMPARACION = pd.DataFrame(FILAS).sort_values("criterio").reset_index(drop=True)
COMPARACION.index += 1; COMPARACION.index.name = "lugar"

# Ensambles de Luz: métricas de validación reportadas en su cuaderno (sin TTA); no hay predicciones por galaxia ni de prueba.
REPORTADOS = pd.DataFrame([
    {"modelo": "Homogéneo de Luz: ResNet50 E (3 semillas, reportado)", "criterio": 0.118602, "RMSE M*": 0.103993, "RMSE SFR": 0.573167, "RMSE D4000": 0.071312,
     "MAE M*": 0.078610, "MAE SFR": 0.380843, "MAE D4000": 0.051014, "R2 M*": 0.974848, "R2 SFR": 0.757310, "R2 D4000": 0.908433,
     "minutos de entrenamiento": 107.05, "detalle": "promedio uniforme de las semillas"},
    {"modelo": "Homogéneo de Luz: ResNet50 D (3 semillas, reportado)", "criterio": 0.119425, "RMSE M*": 0.104255, "RMSE SFR": 0.572519, "RMSE D4000": 0.072495,
     "MAE M*": 0.078730, "MAE SFR": 0.382535, "MAE D4000": 0.051780, "R2 M*": 0.974721, "R2 SFR": 0.757859, "R2 D4000": 0.905369,
     "minutos de entrenamiento": 109.85, "detalle": "pesos por propiedad, λ = 0.01"},
    {"modelo": "Individual de Luz: ResNet50 E semilla 42 (reportado)", "criterio": 0.127879, "RMSE M*": 0.108829, "RMSE SFR": 0.597520, "RMSE D4000": 0.073037,
     "MAE M*": 0.082898, "MAE SFR": 0.394586, "MAE D4000": 0.052342, "R2 M*": 0.972454, "R2 SFR": 0.736249, "R2 D4000": 0.903949,
     "minutos de entrenamiento": 39.63, "detalle": "una red"}]) if PISTA == "con_redshift" else pd.DataFrame()
if len(REPORTADOS):
    REPORTADOS["tipo"] = "reportado (Luz)"
    REPORTADOS["redes por galaxia"] = [3, 3, 1]
    REPORTADOS["ms por galaxia"] = REPORTADOS["redes por galaxia"] * MS_RESNET50
    REPORTADOS["horas de GPU por millón"] = REPORTADOS["ms por galaxia"].map(lambda ms: ensambles.horas_gpu_por_millon(ms) if np.isfinite(ms) else np.nan)
    REPORTADOS["equipo"] = "NVIDIA RTX A4000"
    REPORTADOS["desviación entre repeticiones"] = np.nan
TABLA_VALIDACION = pd.concat([COMPARACION, REPORTADOS], ignore_index=True).sort_values("criterio").reset_index(drop=True)
TABLA_VALIDACION.index += 1; TABLA_VALIDACION.index.name = "lugar"
display(TABLA_VALIDACION[["modelo", "tipo", "criterio", "desviación entre repeticiones", "RMSE M*", "RMSE SFR", "RMSE D4000", "MAE M*", "MAE SFR", "MAE D4000",
                          "R2 M*", "R2 SFR", "R2 D4000"]].round(4).fillna({"desviación entre repeticiones": "no aplica"}))
display(TABLA_VALIDACION[["modelo", "redes por galaxia", "minutos de entrenamiento", "equipo", "ms por galaxia", "horas de GPU por millón", "detalle"]]
        .round(3).fillna("no reportado"))

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(17, 0.9 + 0.5 * len(TABLA_VALIDACION)), sharey=True, gridspec_kw={"width_ratios": [1.3, 1, 1, 1]})
y_pos = np.arange(len(TABLA_VALIDACION))[::-1]
paleta = {"individual": "#8d99ae", "homogéneo": "#2a9d8f", "heterogéneo": "#e76f51", "reportado (Luz)": "#e9c46a"}
colores = [paleta[t] for t in TABLA_VALIDACION.tipo]
ejes[0].barh(y_pos, TABLA_VALIDACION.criterio, color=colores, xerr=TABLA_VALIDACION["desviación entre repeticiones"].fillna(0), capsize=3)
ejes[0].set(yticks=y_pos, yticklabels=TABLA_VALIDACION.modelo, xlabel="MSE normalizado medio (menor es mejor)", title="Métrica principal")
ejes[0].set_xlim(TABLA_VALIDACION.criterio.min() * 0.8, TABLA_VALIDACION.criterio.max() * 1.05)
for eje, t in zip(ejes[1:], OBJETIVOS):
    eje.barh(y_pos, TABLA_VALIDACION[f"RMSE {CORTO[t]}"], color=colores); eje.set(xlabel=f"RMSE {NOMBRES[t]}", title=CORTO[t])
    eje.set_xlim(TABLA_VALIDACION[f"RMSE {CORTO[t]}"].min() * 0.85, TABLA_VALIDACION[f"RMSE {CORTO[t]}"].max() * 1.03)
usados = [t for t in paleta if t in set(TABLA_VALIDACION.tipo)]
fig.legend([plt.Rectangle((0, 0), 1, 1, color=paleta[t]) for t in usados], usados, loc="lower center", ncol=len(usados), frameon=False, fontsize=8)
plt.suptitle(f"Validación ({PISTAS[PISTA]['descripcion']}): modelos ordenados por la métrica principal", y=1.02)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.show()

_Se completa con los resultados de la corrida._

## 7. Selección del modelo final

La métrica principal no es el único criterio. El objetivo del proyecto es extender las propiedades espectroscópicas a galaxias que solo tienen imagen, y eso hace que importen el desempeño, la estabilidad, el costo de usar el modelo en catálogos grandes y los datos que necesita al predecir. Para que la decisión no premie una mejora que no se distingue del ruido a cambio de nueve redes más por galaxia, aplicamos la **regla de un error estándar** que usamos en la fase de ingeniería del Avance 3 (Hastie et al. 2009, §7.10): entre los modelos que no se distinguen del mejor en más de un error estándar, elegimos el más simple, que medimos por el número de redes que hay que ejecutar para cada galaxia.

Adaptamos la regla a que todos los modelos predicen las mismas galaxias. El error estándar que usamos es el de la **diferencia pareada** de la pérdida por galaxia entre cada modelo y el mejor. La versión sin parear mide la dispersión de la pérdida entre galaxias, que crece con las galaxias difíciles para todos los modelos (en la SFR, las galaxias apagadas); cuando los errores de los modelos están correlacionados, como esperamos aquí, esa dispersión es común a todos y puede declarar empatados a modelos que la comparación pareada distingue. Las pérdidas de los ensambles heterogéneos son las de sus predicciones fuera de pliegue, promediadas entre las repeticiones de la validación cruzada. Fijamos esta regla antes de tener las predicciones reales, decidimos antes de abrir el conjunto de prueba y registramos la elección con una huella de sus predicciones de validación: si el cuaderno se volviera a ejecutar con otro modelo elegido, el registro lo detectaría.

In [ ]:
COMPLEJIDAD = dict(zip(COMPARACION.modelo, COMPARACION["redes por galaxia"]))
ELEGIDO, TABLA_REGLA, DETALLE_REGLA = ensambles.regla_un_error_estandar({n: PERDIDAS[n] for n in COMPARACION.modelo}, COMPLEJIDAD)
display(TABLA_REGLA.round(4))
print(f"Mejor criterio: {DETALLE_REGLA['mejor']}\nModelo elegido por la regla: {ELEGIDO}\n"
      f"(Como referencia, el error estándar sin parear del mejor sería {DETALLE_REGLA['error_estandar_sin_parear']:.4f}.)")

In [ ]:
NEGOCIO = COMPARACION[["modelo", "criterio", "desviación entre repeticiones", "redes por galaxia", "horas de GPU por millón"]].copy()
NEGOCIO["entradas al predecir"] = "imagen + redshift espectroscópico" if PISTA == "con_redshift" else "solo imagen"
NEGOCIO["diferencia con el mejor"] = NEGOCIO.modelo.map(TABLA_REGLA["diferencia con el mejor"])
NEGOCIO["error estándar de la diferencia"] = NEGOCIO.modelo.map(TABLA_REGLA["error estándar de la diferencia"])
NEGOCIO["dentro de 1 error estándar"] = NEGOCIO.modelo.map(TABLA_REGLA["dentro de 1 error estándar"])
display(NEGOCIO.round(4).fillna({"desviación entre repeticiones": "no aplica"}).fillna("no reportado"))

_Se completa con los resultados de la corrida._

## 8. Evaluación en el conjunto de prueba

Con el modelo ya elegido y registrado, abrimos el conjunto de prueba **una sola vez**. Los combinadores heterogéneos se ajustaron con toda la validación y se aplicaron sin cambios a las predicciones de prueba de los miembros. Además del modelo final, reportamos los demás modelos de la comparación solo como contexto: sus resultados no pueden cambiar la elección, que ya está registrada.

Como referencia externa incluimos a Katachi (Alfonzo et al. 2024). Sus cifras publicadas (RMSE de 0.22, 0.75 y 0.28 en su conjunto de prueba de SDSS) no se pueden comparar galaxia por galaxia con las nuestras, así que las mostramos como referencia; sus pesos publicados evaluados sobre nuestras galaxias de prueba y la cadena de ResNet50 que reentrenamos con imágenes de DESI en el Avance 3 (media de tres semillas) sí son comparables.

**Independencia de la prueba.** Ninguna decisión de este avance usó el conjunto de prueba. En el Avance 4 sí evaluamos en prueba, una sola vez, las versiones de solo imagen de DINOv3 y ConvNeXt; de ellas tomamos aquí la arquitectura y los hiperparámetros de las versiones con redshift, pero esos hiperparámetros se eligieron en validación antes de aquella evaluación. Las redes con redshift, los combinadores y la elección del modelo final no han visto la prueba.

In [ ]:
RUTA_REGISTRO = CARPETA_ENTREGA / f"registro_eleccion_{PISTA}.json"
REGISTRO = ensambles.registrar_eleccion(RUTA_REGISTRO, {"pista": PISTA, "modelo": ELEGIDO, "commit": COMMIT,
                                                        "criterio_validacion": float(COMPARACION.set_index("modelo").loc[ELEGIDO, "criterio"])},
                                        PRED_VAL[ELEGIDO])
TIPO = dict(zip(COMPARACION.modelo, COMPARACION.tipo))
filas = [{**metricas_largas(n, Y_PRUEBA, PRED_PRUEBA[n], tipo=TIPO[n]), "elegido": n == ELEGIDO} for n in COMPARACION.modelo]
TABLA_PRUEBA = pd.DataFrame(filas).sort_values("criterio").reset_index(drop=True)
display(TABLA_PRUEBA[["modelo", "elegido", "criterio", "RMSE M*", "RMSE SFR", "RMSE D4000", "MAE M*", "MAE SFR", "MAE D4000", "R2 M*", "R2 SFR", "R2 D4000"]].round(4))

REFERENCIAS_KATACHI = pd.DataFrame({"RMSE M*": [0.22, 0.222, 0.223], "RMSE SFR": [0.75, 0.668, 0.653], "RMSE D4000": [0.28, 0.100, 0.073]},
                                   index=["Katachi, artículo (SDSS, otra muestra; no pareable)", "Katachi, pesos publicados en nuestras galaxias de prueba",
                                          "Katachi reentrenado con DESI (Avance 3, media de 3 semillas)"])
display(REFERENCIAS_KATACHI)

_Se completa con los resultados de la corrida._

Para saber si el modelo final es mejor que los demás con una diferencia que no se explique por el azar de la muestra, comparamos con un *bootstrap* pareado: remuestreamos las galaxias de prueba y calculamos en cada remuestreo la diferencia del criterio entre dos modelos. Si el intervalo del 95 % de esa diferencia excluye el cero, la diferencia es significativa. La gráfica muestra las diferencias de la tabla con sus intervalos, y la brecha entre validación y prueba de cada modelo, que indica si generaliza.

In [ ]:
FINAL_PRUEBA = PRED_PRUEBA[ELEGIDO]
filas = []
for nombre in COMPARACION.modelo:
    if nombre == ELEGIDO:
        continue
    d, lo, hi = ensambles.diferencia_pareada_criterio(Y_PRUEBA, FINAL_PRUEBA, PRED_PRUEBA[nombre], ESCALA, N_REMUESTREOS)
    filas.append({"comparado con": nombre, "Δ criterio (final − otro)": d, "límite inferior": lo, "límite superior": hi,
                  "veredicto": "el final es mejor" if hi < 0 else ("el final es peor" if lo > 0 else "sin diferencia significativa")})
PAREADA = pd.DataFrame(filas).set_index("comparado con").sort_values("Δ criterio (final − otro)")
display(PAREADA.round(4))

criterio_val = COMPARACION.set_index("modelo").criterio
criterio_prueba = TABLA_PRUEBA.set_index("modelo").criterio
fig, ejes = plt.subplots(1, 2, figsize=(14, 0.9 + 0.45 * len(COMPARACION)), gridspec_kw={"width_ratios": [1.2, 1]})
y_pos = np.arange(len(PAREADA))[::-1]
ejes[0].errorbar(PAREADA["Δ criterio (final − otro)"], y_pos, xerr=[PAREADA["Δ criterio (final − otro)"] - PAREADA["límite inferior"],
                 PAREADA["límite superior"] - PAREADA["Δ criterio (final − otro)"]], fmt="o", color="#e76f51", capsize=3)
ejes[0].axvline(0, color="k", lw=0.8)
ejes[0].set(yticks=y_pos, yticklabels=PAREADA.index, xlabel="Δ criterio en prueba (negativo: el modelo final es mejor)",
            title=f"Modelo final ({ELEGIDO}) frente a los demás · IC 95 %")
orden = list(criterio_val.sort_values().index)
y2 = np.arange(len(orden))[::-1]
ejes[1].scatter(criterio_val[orden], y2, color="0.5", label="validación", zorder=3)
ejes[1].scatter(criterio_prueba[orden], y2, color="#264653", label="prueba", zorder=3)
for yy, m in zip(y2, orden):
    ejes[1].plot([criterio_val[m], criterio_prueba[m]], [yy, yy], color="0.75", lw=1)
ejes[1].set(yticks=y2, yticklabels=[("★ " if m == ELEGIDO else "") + m for m in orden], xlabel="criterio (menor es mejor)", title="Brecha entre validación y prueba")
ejes[1].legend(fontsize=7); plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

## 9. Gráficos del modelo final

Como nuestro problema es de regresión, las curvas ROC y las matrices de confusión no aplican; usamos los gráficos propios de la regresión. Para el modelo final y sobre las galaxias de prueba, que ningún paso del proceso había visto, mostramos la concordancia entre predicción y etiqueta, la distribución de los residuos, cómo cambia el error con el valor de la propiedad y con el redshift, y si el desacuerdo entre familias sirve como señal de incertidumbre. Cerramos con lo que aporta cada familia al modelo final, medido en validación por el cuaderno de ensambles heterogéneos. En los gráficos de residuos comparamos con la mejor red individual.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4.4))
for k, (eje, t) in enumerate(zip(ejes, OBJETIVOS)):
    y, p = Y_PRUEBA[:, k], FINAL_PRUEBA[:, k]
    lim = np.percentile(np.r_[y, p], [0.5, 99.5])
    eje.hexbin(y, p, gridsize=45, cmap="viridis", mincnt=1, bins="log", extent=(*lim, *lim)); eje.plot(lim, lim, "k-", lw=0.8)
    m = evaluacion.metricas(y, p)
    eje.text(0.03, 0.97, f"RMSE {m['RMSE']:.3f}\nR² {m['R2']:.3f}\nsesgo {m['sesgo']:+.3f}\npendiente {ensambles.pendiente_de_calibracion(y, p):.2f}",
             transform=eje.transAxes, va="top", fontsize=8, bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="0.7"))
    eje.set(xlim=lim, ylim=lim, xlabel=f"{NOMBRES[t]} (etiqueta)", ylabel="predicción")
plt.suptitle(f"Prueba: predicción frente a etiqueta · {ELEGIDO}", y=1.02); plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 3.8))
for k, (eje, t) in enumerate(zip(ejes, OBJETIVOS)):
    for nombre, color, etiqueta in ((NOMBRE_INDIVIDUAL, "0.55", "mejor red individual"), (ELEGIDO, "#e76f51", "modelo final")):
        r = PRED_PRUEBA[nombre][:, k] - Y_PRUEBA[:, k]
        nmad = evaluacion.metricas(Y_PRUEBA[:, k], PRED_PRUEBA[nombre][:, k])["sigma_NMAD"]
        eje.hist(r, bins=60, density=True, histtype="step", lw=1.6, color=color, label=f"{etiqueta}: sesgo {r.mean():+.3f}, σ_NMAD {nmad:.3f}")
    eje.axvline(0, color="k", lw=0.6); eje.set(xlabel=f"residuo de {NOMBRES[t]} (predicción − etiqueta)", ylabel="densidad", title=CORTO[t]); eje.legend(fontsize=7)
plt.suptitle("Prueba: distribución de los residuos", y=1.02); plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
def mediana_por_intervalos(x, residuo, n=10):
    n = int(max(2, min(n, len(x) // 12)))
    bordes = np.quantile(x, np.linspace(0, 1, n + 1)); centro, med, p16, p84 = [], [], [], []
    for a, b in zip(bordes[:-1], bordes[1:]):
        s = (x >= a) & (x <= b)
        if s.sum() >= 4:
            centro.append(np.median(x[s])); med.append(np.median(residuo[s])); p16.append(np.percentile(residuo[s], 16)); p84.append(np.percentile(residuo[s], 84))
    return map(np.array, (centro, med, p16, p84))

fig, ejes = plt.subplots(2, 3, figsize=(14, 7))
for k, t in enumerate(OBJETIVOS):
    for fila, (x, nombre_x) in enumerate(((Y_PRUEBA[:, k], f"{NOMBRES[t]} (etiqueta)"), (Z_PRUEBA, "redshift medido"))):
        for nombre, color, etiqueta in ((NOMBRE_INDIVIDUAL, "0.55", "mejor red individual"), (ELEGIDO, "#e76f51", "modelo final")):
            cx, med, p16, p84 = mediana_por_intervalos(x, PRED_PRUEBA[nombre][:, k] - Y_PRUEBA[:, k])
            ejes[fila, k].plot(cx, med, "o-", ms=3, color=color, label=etiqueta); ejes[fila, k].fill_between(cx, p16, p84, color=color, alpha=0.12)
        ejes[fila, k].axhline(0, color="k", lw=0.6); ejes[fila, k].set(xlabel=nombre_x, ylabel="mediana del residuo")
    ejes[0, k].set_title(NOMBRES[t])
ejes[0, 0].legend(fontsize=7)
plt.suptitle("Prueba: residuo según el valor de la propiedad (arriba) y según el redshift (abajo); franja: 68 % central", y=1.0)
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
# Desacuerdo entre familias como señal de incertidumbre: ¿los casos en que los miembros discrepan son los que más fallan?
DESACUERDO = ensambles.desacuerdo_ponderado(P_PRUEBA, PESOS_H1 if ELEGIDO == NOMBRE_H1 else None)
ERROR_ABSOLUTO = np.abs(FINAL_PRUEBA - Y_PRUEBA)
fig, ejes = plt.subplots(1, 3, figsize=(14, 3.9))
for k, (eje, t) in enumerate(zip(ejes, OBJETIVOS)):
    cuantiles = np.quantile(DESACUERDO[:, k], np.linspace(0, 1, 6))
    grupo = np.clip(np.searchsorted(cuantiles, DESACUERDO[:, k], side="right") - 1, 0, 4)
    centros = [np.median(DESACUERDO[grupo == g, k]) for g in range(5)]
    medianas = [np.median(ERROR_ABSOLUTO[grupo == g, k]) for g in range(5)]
    p84 = [np.percentile(ERROR_ABSOLUTO[grupo == g, k], 84) for g in range(5)]
    rho = spearmanr(DESACUERDO[:, k], ERROR_ABSOLUTO[:, k]).statistic
    eje.hexbin(DESACUERDO[:, k], ERROR_ABSOLUTO[:, k], gridsize=35, cmap="Greys", mincnt=1, bins="log")
    eje.plot(centros, medianas, "o-", color="#e76f51", label="mediana del error por quintil"); eje.plot(centros, p84, "s--", color="#264653", label="percentil 84")
    eje.set(xlabel="desacuerdo entre familias", ylabel="error absoluto", title=f"{NOMBRES[t]} · ρ de Spearman = {rho:.2f}"); eje.legend(fontsize=7)
plt.suptitle("Prueba: ¿sirve el desacuerdo entre familias como señal de incertidumbre?", y=1.02); plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
# Aporte de cada familia al modelo final, calculado en validación por el cuaderno de ensambles heterogéneos.
if TIPO[ELEGIDO] == "heterogéneo":
    a = APORTE[APORTE.ensamble == ELEGIDO].set_index("familia").reindex(FAMILIAS_HETEROGENEO)
    fig, ejes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.4, 1]})
    ancho = 0.8 / len(FAMILIAS_HETEROGENEO)
    for i, f in enumerate(FAMILIAS_HETEROGENEO):
        ejes[0].bar(np.arange(3) + i * ancho, [a.loc[f, f"aumento {CORTO[t]}"] for t in OBJETIVOS], ancho * 0.9, label=f)
    ejes[0].axhline(0, color="k", lw=0.6)
    ejes[0].set(xticks=np.arange(3) + ancho * (len(FAMILIAS_HETEROGENEO) - 1) / 2, xticklabels=[NOMBRES[t] for t in OBJETIVOS],
                ylabel="aumento del MSE normalizado al quitar la familia", title="Aporte de cada familia por propiedad"); ejes[0].legend(fontsize=7)
    if ELEGIDO == NOMBRE_H1:
        ejes[1].imshow(PESOS_H1.T, cmap="viridis", vmin=0, vmax=1, aspect="auto")
        ejes[1].set(xticks=range(len(FAMILIAS_HETEROGENEO)), yticks=range(3), yticklabels=[CORTO[t] for t in OBJETIVOS], title="Pesos de la mezcla")
        ejes[1].set_xticklabels(FAMILIAS_HETEROGENEO, rotation=45, ha="right", fontsize=7)
        for i in range(len(FAMILIAS_HETEROGENEO)):
            for j in range(3):
                ejes[1].text(i, j, f"{PESOS_H1[i, j]:.2f}", ha="center", va="center", fontsize=8, color="w" if PESOS_H1[i, j] < 0.5 else "k")
    else:
        ejes[1].barh(range(len(FAMILIAS_HETEROGENEO)), a["aumento del criterio"], color="#e76f51")
        ejes[1].set(yticks=range(len(FAMILIAS_HETEROGENEO)), yticklabels=FAMILIAS_HETEROGENEO, xlabel="aumento del criterio global", title="Aporte global")
    plt.suptitle("Qué aporta cada arquitectura al modelo final (validación, sin la familia)", y=1.02); plt.tight_layout(); plt.show()
else:
    print("El modelo final no es un ensamble heterogéneo: no hay aportes por familia que mostrar.")

_Se completa con los resultados de la corrida._

## 10. Conclusiones

_Se completa con los resultados de la corrida._

## 11. Referencias

Alfonzo, J. P., Iyer, K. G., Akiyama, M., et al. (2024). Decoding the imprints of past star formation on present-day morphology in galaxies with interpretable CNNs. *The Astrophysical Journal*, 967, 152.

Breiman, L. (1996). Stacked regressions. *Machine Learning*, 24(1), 49–64.

Cawley, G. C. y Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research*, 11, 2079–2107.

Hastie, T., Tibshirani, R. y Friedman, J. (2009). *The Elements of Statistical Learning* (2.ª ed.). Springer. Sección 7.10.

Krogh, A. y Vedelsby, J. (1995). Neural network ensembles, cross validation, and active learning. *Advances in Neural Information Processing Systems 7*, 231–238.

Singh, A. (2023, 14 de julio). Comprehensive guide to ensemble learning (with Python codes). *Analytics Vidhya*. https://www.analyticsvidhya.com/blog/2018/06/comprehensive-guide-for-ensemble-models/

Studer, S., Bui, T. B., Drescher, C., et al. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. *Machine Learning and Knowledge Extraction*, 3, 392–413.

VanderPlas, J. (2022). *Python Data Science Handbook* (2.ª ed.). O'Reilly Media. Capítulo 39, sección *Thinking about model validation*.

Wolpert, D. H. (1992). Stacked generalization. *Neural Networks*, 5(2), 241–259.

Zhong, J., Deng, Z., Li, X., Wang, L., Yang, H., Li, H. y Zhao, X. (2024). Estimation of stellar mass and star formation rate based on galaxy images. *Monthly Notices of the Royal Astronomical Society*, 531(1), 2011–2027.

# Declaración de uso de IA y contribución del equipo

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (celdas o secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude [Modelo de lenguaje grande]. https://claude.ai/ | Escribir el código de los cuadernos de modelos base, de ensambles heterogéneos y de comparación final, con sus pruebas unitarias, y redactar los análisis a partir de los resultados | Este cuaderno, el de ensambles heterogéneos y el de entrenamiento de los modelos base | Ejecutamos los entrenamientos y los ensambles, comparamos cada cifra con las salidas de los cuadernos, corregimos el código y los textos, y validamos cada interpretación |
| OpenAI. (2026). ChatGPT (GPT-6) [Modelo de lenguaje grande]. https://chatgpt.com/ | Estructurar el código y revisar el procedimiento de comparación en los modelos 1, 2, 5 y 6 y en los ensambles homogéneos | Cuadernos de los modelos 1, 2, 5 y 6 y de los ensambles homogéneos, de los que este cuaderno toma las métricas de referencia de ResNet50 | Revisamos las salidas antes de interpretarlas y adaptamos el código a las variables astronómicas, con la orientación de los asesores |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Luz Copelia Minutti Pérez | Desarrolló y ejecutó los modelos 1 (ResNet50 multitarea), 2 (SVR), 5 (Ridge) y 6 (KNN), el protocolo común de comparación y los ensambles homogéneos de ResNet50 (variantes D y E), con la búsqueda de sus pesos y su análisis de importancia de las entradas. |
| Erick Alberto Bustos Cruz | Desarrolló y ejecutó los modelos 3 (ConvNeXt-Tiny) y 4 (DINOv3 con LoRA), entrenó los modelos base con redshift y EfficientNet-B3, construyó los ensambles heterogéneos y redactó la comparación, la selección y la evaluación del modelo final. |
| José Florencio Maguey Peralta | _Pendiente: el integrante debe completar su contribución._ |

### Decisiones principales

<small><b>Decisión 1:</b> Usar la validación como conjunto de reserva de los ensambles heterogéneos, con validación cruzada anidada</small>
- <small>Justificación: reentrenar cada red varias veces sobre pliegues del entrenamiento costaba horas de GPU por red, y la validación cruzada anidada permite medir el desempeño de los combinadores sin que las galaxias evaluadas hayan intervenido en su ajuste.</small>
- <small>Alternativa descartada y motivo: stacking con predicciones por pliegues sobre todo el entrenamiento, porque exigía entrenar de nuevo cada red varias veces (unas 2 horas con DINOv3 y unas 3 con ResNet50).</small>

<small><b>Decisión 2:</b> Construir los ensambles en dos pistas, con y sin el redshift espectroscópico, controladas por una sola variable</small>
- <small>Justificación: el redshift espectroscópico mejora mucho la masa, pero no existe para la mayoría de las galaxias que solo tienen imagen, y el uso final del modelo determina cuál pista conviene.</small>
- <small>Alternativa descartada y motivo: elegir una sola pista desde el inicio, porque habría ocultado cuánto se pierde al no conocer la distancia.</small>

<small><b>Decisión 3:</b> Elegir el modelo final con la regla de un error estándar pareada, y que el meta-aprendiz corrija el promedio uniforme</small>
- <small>Justificación: evita pagar el costo de ejecutar muchas redes por una mejora que no se distingue del ruido, y mantiene al meta-aprendiz cerca de un promedio razonable aunque haya solo 1,261 galaxias de reserva.</small>
- <small>Alternativa descartada y motivo: elegir el modelo con el menor criterio sin más, y un meta-aprendiz libre sobre las predicciones, porque puede sobreajustar con tan pocas galaxias.</small>